# Data Enrichment

## Loading Packages

In [1]:
import numpy as np
import pandas as pd
import glob
import plotly.express as px
import folium

import os
from pathlib import Path
from urllib.request import urlretrieve
from urllib.error import HTTPError, URLError
from zipfile import ZipFile

In [2]:
output_dir = "../data"

In [3]:
citibike_df = pd.read_csv(f"{output_dir}/JC/JC2025.csv", parse_dates=['started_at', 'ended_at'])

## Duration

In [4]:
citibike_df['ride_duration_min'] = (citibike_df["ended_at"] - citibike_df["started_at"]).dt.total_seconds()/60

In [5]:
citibike_df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,ride_duration_min
0,880A0159BA5275FB,electric_bike,2025-01-16 17:50:49.136,2025-01-16 17:57:00.710,Hilltop,JC019,Pershing Field,JC024,40.731169,-74.057574,40.742677,-74.051789,member,6.192900
1,1A5E1E274B2AF0AD,electric_bike,2025-01-31 06:10:41.818,2025-01-31 06:22:09.499,Hilltop,JC019,Jackson Square,JC063,40.731169,-74.057574,40.711130,-74.078900,member,11.461350
2,EA9928D3C05B8377,classic_bike,2025-01-09 16:42:50.213,2025-01-09 17:04:12.870,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,40.735938,-74.030305,member,21.377617
3,3C42C367750B9292,electric_bike,2025-01-21 16:14:14.398,2025-01-21 16:37:10.458,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,40.735938,-74.030305,member,22.934333
4,94D3B0265A7BDE1F,classic_bike,2025-01-30 16:38:18.840,2025-01-30 17:04:08.166,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,40.735938,-74.030305,member,25.822100


In [6]:
citibike_df.shape

(1002704, 14)

In [7]:
citibike_df = citibike_df[
    (citibike_df['ride_duration_min']>1) & (citibike_df['ride_duration_min']<24*60)
]

citibike_df.shape

(1002333, 14)

## Missing Values

In [8]:
missing_values = citibike_df.isnull().sum().reset_index()

missing_values.columns = ['columns_name', 'missing_count']

missing_values['missing_percentage'] = (missing_values['missing_count'] / len(citibike_df)) * 100

missing_values.sort_values(by='missing_percentage', ascending=False, inplace=True)

missing_values

,columns_name,missing_count,missing_percentage
7,end_station_id,4049,0.403958
10,end_lat,3106,0.309877
11,end_lng,3106,0.309877
6,end_station_name,2887,0.288028
4,start_station_name,3,0.000299
5,start_station_id,3,0.000299
9,start_lng,2,0.000200
8,start_lat,2,0.000200
0,ride_id,0,0.000000
1,rideable_type,0,0.000000


## Droping Missing Values

In [9]:
citibike_df.columns

Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual', 'ride_duration_min'],
      dtype='str')

In [10]:
citibike_df = citibike_df.dropna(
    subset=[
        'ride_id', 'started_at', 'ended_at',
        'start_lat', 'start_lng', 'end_lat', 'end_lng',
        'end_station_id', 'end_station_name', 
        'start_station_name', 'start_station_id'
    ]
)

## Adding Time Granularity

In [11]:
citibike_df['date'] = citibike_df['started_at'].dt.date
citibike_df['month'] = citibike_df['started_at'].dt.to_period('M').astype(str)
citibike_df['month_name'] = citibike_df['started_at'].dt.month_name()
citibike_df['day_of_week'] = citibike_df['started_at'].dt.day_name()
citibike_df['hour'] = citibike_df['started_at'].dt.hour
citibike_df['month_number'] = citibike_df['started_at'].dt.month

In [12]:
citibike_df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,ride_duration_min,date,month,month_name,day_of_week,hour,month_number
0,880A0159BA5275FB,electric_bike,2025-01-16 17:50:49.136,2025-01-16 17:57:00.710,Hilltop,JC019,Pershing Field,JC024,40.731169,-74.057574,40.742677,-74.051789,member,6.192900,2025-01-16,2025-01,January,Thursday,17,1
1,1A5E1E274B2AF0AD,electric_bike,2025-01-31 06:10:41.818,2025-01-31 06:22:09.499,Hilltop,JC019,Jackson Square,JC063,40.731169,-74.057574,40.711130,-74.078900,member,11.461350,2025-01-31,2025-01,January,Friday,6,1
2,EA9928D3C05B8377,classic_bike,2025-01-09 16:42:50.213,2025-01-09 17:04:12.870,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,40.735938,-74.030305,member,21.377617,2025-01-09,2025-01,January,Thursday,16,1
3,3C42C367750B9292,electric_bike,2025-01-21 16:14:14.398,2025-01-21 16:37:10.458,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,40.735938,-74.030305,member,22.934333,2025-01-21,2025-01,January,Tuesday,16,1
4,94D3B0265A7BDE1F,classic_bike,2025-01-30 16:38:18.840,2025-01-30 17:04:08.166,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,40.735938,-74.030305,member,25.822100,2025-01-30,2025-01,January,Thursday,16,1


## Adding Seasons

In [13]:
def assign_season(month_number):
    if month_number in [12,1,2]:
        return 'Winter'
    elif month_number in[3,4,5]:
        return 'Spring'
    elif month_number in [6,7,8]:
        return 'Summer'
    else:
        return 'Autumn'

In [14]:
citibike_df['season'] = citibike_df['month_number'].apply(assign_season)

citibike_df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,...,end_lng,member_casual,ride_duration_min,date,month,month_name,day_of_week,hour,month_number,season
0,880A0159BA5275FB,electric_bike,2025-01-16 17:50:49.136,2025-01-16 17:57:00.710,Hilltop,JC019,Pershing Field,JC024,40.731169,-74.057574,...,-74.051789,member,6.192900,2025-01-16,2025-01,January,Thursday,17,1,Winter
1,1A5E1E274B2AF0AD,electric_bike,2025-01-31 06:10:41.818,2025-01-31 06:22:09.499,Hilltop,JC019,Jackson Square,JC063,40.731169,-74.057574,...,-74.078900,member,11.461350,2025-01-31,2025-01,January,Friday,6,1,Winter
2,EA9928D3C05B8377,classic_bike,2025-01-09 16:42:50.213,2025-01-09 17:04:12.870,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,...,-74.030305,member,21.377617,2025-01-09,2025-01,January,Thursday,16,1,Winter
3,3C42C367750B9292,electric_bike,2025-01-21 16:14:14.398,2025-01-21 16:37:10.458,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,...,-74.030305,member,22.934333,2025-01-21,2025-01,January,Tuesday,16,1,Winter
4,94D3B0265A7BDE1F,classic_bike,2025-01-30 16:38:18.840,2025-01-30 17:04:08.166,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,...,-74.030305,member,25.822100,2025-01-30,2025-01,January,Thursday,16,1,Winter


## Storing Enriched Data

In [15]:
citibike_df.to_csv(f'{output_dir}/JC/JC2025_Enriched.csv', index=False)